### Data Parser Notebook

This notebook contains code to parse daily stock data from raw price data into change over time data for training.

In [1]:
from datetime import date, timedelta
import pandas as pd
from pandas import DataFrame
import numpy as np
import math

DATE_LEN = 10
DATA_POINTS_PER_TICKER = 75
CSV_HEADER_OFFSET = 2
MAX_RAND = 4

In [2]:
def parse_date(date_str:str) -> date:
    items = date_str.split('-')[:DATE_LEN]

    return date(year=int(items[0]), month=int(items[1]), day=int(items[2]))

def format_date(date: date) -> str:
    year, month, day = date.year, date.month, date.day

    return f"{year}-{month}-{day}"

In [3]:
def calculate_volatility(df: DataFrame, price_col: str, dates: DataFrame, start_index: int, end_index: int, log:bool= True) -> DataFrame:
    # Calculate log returns
    float_range = df[price_col][start_index:end_index].astype('float64')

    no_nan_range = float_range.fillna(np.mean(float_range))

    log_returns = np.log((no_nan_range / no_nan_range.shift(1)))

    # Calculate annualized volatility (252 trading days)
    volatility = log_returns.std() * np.sqrt((parse_date(dates['Price'][end_index]) - parse_date(dates['Price'][start_index])).days)

    if log: print(f"Annualized Volatility: {volatility:.2%}")

    return volatility

In [4]:
# Generates a data point from a ticker and the date of the label
# Train Len is set to 252 days (one TSX trading year) by default
# Offset Len is set to 126 days (half TSX trading year) by default
def generate_data_point(df: DataFrame, price_col: str, volume_col: str, dates: DataFrame, label_index: int, offset_len: int = 126, train_len: int = 252, log = True) -> list[float]:
    fend = label_index - offset_len
    fstart  = fend - train_len

    # One extra day for prices to calculate initial gain, fill NaNs forward
    raw_prices = df[price_col][(fstart-1):fend].astype('float64')

    if log: print("Raw prices:", raw_prices)

    feature_changes = list((raw_prices - raw_prices.shift(1)) / raw_prices.shift(1))[1:]

    if log: print("% Changes:", feature_changes)

    raw_volumes = df[volume_col][fstart:fend].astype('float64')

    no_nan_volumes = raw_volumes.fillna(np.mean(raw_volumes))

    if log: print("No NaN Volumes:", no_nan_volumes)

    # Mean center the volumes
    # Done to remove raw volume as a factor of training, focus more on changes of volume with price swings
    # As a lot of the data points have drastically differing volumes
    feature_volumes = (no_nan_volumes - np.mean(no_nan_volumes)) / np.std(no_nan_volumes)

    if log: print("Volumes:", feature_volumes)
    
    label_price = float(df[price_col][label_index])

    if log: print("Label Price:", label_price)

    index = 1
    feature_end = raw_prices.tail(index).iloc[0]

    if log: print("Last Feature Price:", feature_end)
    
    label_change = (label_price - feature_end) / feature_end

    if log: print("Change Label:", label_change)

    label_volat = calculate_volatility(df, price_col, dates, fend, label_index, log=log)

    #if math.isnan(label_change):
        #raise Exception("NaN label change", label_index, label_price, feature_end)

    assert(len(feature_volumes) == len(feature_changes))

    return list(feature_changes) + list(feature_volumes) +  [ label_change, label_volat ]

In [5]:
# Useless LOL
def df_price_to_prcnts(df: DataFrame, price_col: str, log: bool = True) -> list[float]:
    output_raw = []
    df_tuples = df.itertuples()
    last = df_tuples.__next__()

    for row in df_tuples:
        avg_dod = (float(getattr(row, price_col)) - float(getattr(last, price_col))) / (float(getattr(last, price_col)))

        ## Over the weekends, price data is extrapolated to fill in missing data points
        #if days_since_last > 1:
        #    for _ in range(days_since_last):
        #        output_raw.append(avg_dod)
        #else:
        #   output_raw.append(avg_dod)

        output_raw.append(avg_dod)

        last = row

    if log: print("DF Price to Prcnts head:", output_raw[:10])

    return output_raw

In [6]:
df = pd.read_csv('raw_phist.csv')

dates = df.filter(axis=1, like='Price')
prices = df.filter(axis=1, like='Close')
volumes = df.filter(axis=1, like='Volume')

C:\Users\austi\AppData\Local\Temp\ipykernel_17900\2070281477.py:1: DtypeWarning: Columns (0: Close, 1: Close.1, 2: Close.2, 3: Close.3, 4: Close.4, 5: Close.5, 6: Close.6, 7: Close.7, 8: Close.8, 9: Close.9, 10: Close.10, 11: Close.11, 12: Close.12, 13: Close.13, 14: Close.14, 15: Close.15, 16: Close.16, 17: Close.17, 18: Close.18, 19: Close.19, 20: Close.20, 21: Close.21, 22: Close.22, 23: Close.23, 24: Close.24, 25: Close.25, 26: Close.26, 27: Close.27, 28: Close.28, 29: Close.29, 30: Close.30, 31: Close.31, 32: Close.32, 33: Close.33, 34: Close.34, 35: Close.35, 36: Close.36, 37: Close.37, 38: Close.38, 39: Close.39, 40: Close.40, 41: Close.41, 42: Close.42, 43: Close.43, 44: Close.44, 45: Close.45, 46: Close.46, 47: Close.47, 48: Close.48, 49: Close.49, 50: Close.50, 51: Close.51, 52: Close.52, 53: Close.53, 54: Close.54, 55: Close.55, 56: Close.56, 57: Close.57, 58: Close.58, 59: Close.59, 60: Close.60, 61: Close.61, 62: Close.62, 63: Close.63, 64: Close.64, 65: Close.65, 66: Clos

In [13]:
def generate_data_points(df: DataFrame, dates: DataFrame, price_col: str, volume_col: str, amount: int, log: bool = True) -> list[list[float]]:    
    price_data = df[price_col][CSV_HEADER_OFFSET:].ffill()
    start_valid = price_data.first_valid_index()
    end_valid = price_data.last_valid_index()

    if log: print("Valid Date Range:", df['Price'][start_valid], "|", df['Price'][end_valid])

    exclude = set({})

    TSX_YEAR = 252

    output = []

    for _ in range(amount):
        if start_valid + TSX_YEAR * 1.5 + 1 >= end_valid:
            if log: print("No range available for training, returning empty")
            return []

        label_index = np.random.randint(start_valid + TSX_YEAR * 1.5 + 1, end_valid)

        tries = 0

        while (label_index in exclude or math.isnan(float(price_data[:][label_index]))) and tries < MAX_RAND:
            label_index = np.random.randint(start_valid + TSX_YEAR * 1.5 + 1, end_valid)
            tries += 1

            if tries == MAX_RAND:
                if log: print("Random exceeded max tries, returning early")
                return output
        
        exclude.add(label_index)

        point = generate_data_point(df, price_col, volume_col, dates, label_index, log=log)

        if log: print("Point:", point)

        if np.count_nonzero(np.isnan(point)) == 0:
            output.append(point)

    return output

In [14]:
data = []

log = False

for (price, volume) in zip(prices, volumes):
    point_data = generate_data_points(df, dates, price, volume, DATA_POINTS_PER_TICKER, log=log) 

    if np.count_nonzero(np.isnan(point_data)) > 0:
        #print("Data parsing failed for:", price, volume)
        #print(point_data)
        continue

    data = data + point_data

c:\Users\austi\OneDrive\Documents\School\3ML3\FinalProject\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\austi\OneDrive\Documents\School\3ML3\FinalProject\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\austi\OneDrive\Documents\School\3ML3\FinalProject\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [15]:
print(np.count_nonzero(np.isnan(data)))

0


In [16]:
import os

if os.path.exists('stock_curated.csv'): os.remove('stock_curated.csv')

np.savetxt('stock_curated.csv', np.array(data).T, delimiter=',')

In [17]:
np.array(data).shape

(43164, 506)